In [1]:
#定义奖励模型
import torch
from typing import Optional
from torch import nn
import numpy as np
import os
from sympy import false
from transformers import AutoModelForCausalLM, AutoTokenizer

class RewardHead(nn.Module):
    """
    RewardHead类给GPT2实现了一个“头”，为每个输出的token返回一个标量值。
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(
            self.reward.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.reward(output)


class GPT2RewardModel(nn.Module):
    """
    GPT2模型加上一个“奖励头”
    """

    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        # 添加奖励头
        self.reward_head = RewardHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # GPT2的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )

        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 对隐藏层给出奖励
        rewards = self.reward_head(last_hidden_state).squeeze(-1)
        # 归一化
        return torch.sigmoid(rewards)

#加载模型
model_path = "C:\\Users\lihaodong\.cache\modelscope\hub\models\Fengshenbang\Wenzhong-GPT2-110M-chinese-v2"
model = GPT2RewardModel(model_path)
if os.path.exists('models/best_reward_model.pt'):
    model.load_state_dict(torch.load('models/best_reward_model.pt'))
    print('reward_model.pt loaded')



D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.regi

In [2]:
#将最开始的SFT和价值模型训练代码放在一起，价值模型就是SFT后面训练一个价值头，和奖励模型一样，价值模型就是评论家，奖励模型就是裁判。SFT训练好后，价值模型和奖励模型都要保存下来，以便后续PPO训练使用。
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class ValueHead(nn.Module):
    """
    ValueHead类为GPT2实现了一个“头”，会为输出的每个token返回一个标量值
    标量值就是这个token的价值，ValueHead就是评论家。
    """
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.value = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(
            self.value.weight,
            std=(1.0 / np.sqrt(self.hidden_size + 1))
        )
        nn.init.zeros_(self.value.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.value(output)

In [3]:
#整体模型
class ModelForCausalLMWithValueHead(nn.Module):
    """
    GPT2模型+一个价值头
    """
    def __init__(self, model_path):
        super().__init__()
        # 这个要初始化为我们微调出来的gpt2-sft模型
        # actor演员模型
        self.llm = AutoModelForCausalLM.from_pretrained(model_path)
        # 添加价值头
        # critic评论家模型
        self.v_head = ValueHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # gpt2-sft模型的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )
        # 输出的token的概率分布，维度为 `vocab_size`
        lm_logits = transformer_outputs.logits
        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 评估token的价值
        value = self.v_head(last_hidden_state).squeeze(-1)
        # 返回输出的token的logits和token的价值
        return lm_logits, value

    def generate(self, *args, **kwargs):
        return self.llm.generate(*args, **kwargs)

In [12]:
#初始化模型，加载微调好的gpt2-sft模型，后续PPO训练的时候就会在这个基础上继续训练价值头。
model_path = '../SFT微调大模型/models/fine_tuned_gpt2'
model = ModelForCausalLMWithValueHead(model_path)
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)
# model.eval()
# prompt = "今天天气真好，我想去"   # 可以修改成你想要的任何开头
# inputs = tokenizer(prompt, return_tensors='pt')
# input_ids = inputs['input_ids']
# attention_mask = inputs['attention_mask']
#
# with torch.no_grad():
#     # 设置生成参数
#     outputs = model.generate(
#         input_ids=input_ids,
#         attention_mask=attention_mask,
#         max_new_tokens=100,          # 生成的新 token 数量，你可以调大
#         do_sample=True,              # 启用采样（随机性）
#         temperature=0.01,             # 控制随机性，值越小越保守
#         top_p=0.9,                   # nucleus sampling
#         repetition_penalty=1.1,      # 避免重复
#         pad_token_id=tokenizer.pad_token_id,
#         eos_token_id=tokenizer.eos_token_id,
#     )
#
# generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print("生成的文本：")
# print(generated_text)

生成的文本：
今天天气真好，我想去看一个电影可是我没有看过这部电影但是我觉得还是不错看完之后我觉得还是不错看完之后我觉得还是不错看完之后我�


In [ ]:
import random
#准备数据集
from functools import partial
#加载tokenizer
from datasets import load_dataset
from transformers import AutoTokenizer


#定义collate
def reward_collate_fn(batch):
    # batch 是一个列表，每个元素是字典，包含 'input_ids', 'attention_mask', 'score', 'score_index' 等
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    scores = [item['score'] for item in batch]
    # 原有的 score_index 不再使用，直接重新计算：
    # 每个样本的 reward token 在原始序列中的位置就是最后一个 token
    # 但经过 padding 后，我们需要找到每个样本的最后一个有效 token 的位置
    # 方法：先 padding，然后根据 attention_mask 找到每个样本的最后一个 1 的位置
    # 注意：padding 会改变序列长度，但我们可以在 padding 后再计算。

    # 使用 tokenizer 进行 padding
    padded = tokenizer.pad(
        {'input_ids': input_ids, 'attention_mask': attention_mask},
        padding=True,
        return_tensors='pt'
    )

    # 计算每个样本的最后一个有效 token 的索引
    # attention_mask 是 (batch_size, seq_len)，每行是 0/1
    # 使用 torch.argmax 可以找到最后一个 1 的位置（因为 argmax 返回第一个最大值的索引，对于 0/1 序列，最后一个 1 的位置需要更复杂的方法）
    # 简便方法：取每行非零位置的最大索引
    seq_len = padded['attention_mask'].shape[1]
    # 创建一个索引矩阵
    indices = torch.arange(seq_len).unsqueeze(0).expand(padded['attention_mask'].shape[0], -1).to(padded['attention_mask'].device)
    # 将 attention_mask 为 0 的位置索引设为 -1
    masked_indices = indices * padded['attention_mask']
    # 取每行的最大值，就是最后一个有效 token 的索引
    score_index = masked_indices.max(dim=1).values

    # 将 scores 转换为 tensor
    scores_tensor = torch.tensor(scores, dtype=torch.float32)

    # 返回一个字典，包含 input_ids, attention_mask, score, score_index
    return {
        'input_ids': padded['input_ids'],
        'attention_mask': padded['attention_mask'],
        'score': scores_tensor,
        'score_index': score_index
    }

# 定义分词函数：接收一个 batch 字典，返回一个字典（键是新的列名，值是列表）
def tokenize(sample, tokenizer,input_token_length_range):
    input_size = random.choice(input_token_length_range)
    sample['input_ids'] = tokenizer.encode(sample['context'])[:input_size]
    sample['attention_mask'] = [1] * len(sample['input_ids'])
    sample['query'] = tokenizer.decode(sample['input_ids'])
    return sample


#清理数据长度国小的
def filter_short_samples(batch, min_length=8):
    # 计算输入 ID 的长度
    input_ids_length = len(batch['input_ids'])
    # 只有当长度大于或等于 min_length和小于等于100 时才保留样本
    return (input_ids_length >= min_length and input_ids_length <= 150)

def load_data(data_path,tokenizer):
    dataset = load_dataset('json', data_files={
        'train': data_path[0],
        'valid': data_path[1],
        'test': data_path[2]
    })
    #过滤掉长度过短的样本，避免训练时出现过多无效样本
    dataset["train"] = dataset["train"].filter(filter_short_samples)
    dataset["valid"] = dataset["valid"].filter(filter_short_samples)

    #控制输出数量
    import random
    input_min_token_length = 3
    input_max_token_length = 20
    input_token_length_range = list(range(
        input_min_token_length,
        input_max_token_length))
    print(input_token_length_range)
    print(random.choice(input_token_length_range))

    # 查看结构
    print(dataset['train'][0])
    # 输出：{'idx': 0, 'context': '如果 我 无聊 时 ...', 'label': 1}

    map_kwargs = {
        'batched': True,  # 启用批处理模式，tokenize 接收的是 batch 字典
        'batch_size': 512,  # 每批 512 个样本
        'remove_columns': ['idx', 'context', 'label']  # 处理后移除这三列
    }

    # 在 map 时绑定 tokenizer
    tokenize_with_tokenizer = partial(tokenize, tokenizer=tokenizer,input_token_length_range=input_token_length_range)

    # 对训练集和验证集应用 map
    tokenized_dataset_train = dataset["train"].map(tokenize_with_tokenizer, **map_kwargs)
    tokenized_dataset_val = dataset["valid"].map(tokenize_with_tokenizer, **map_kwargs)

    print(tokenized_dataset_train[0])
    print(tokenized_dataset_val[0])

    tokenized_dataset_train.set_format(type='torch')
    tokenized_dataset_val.set_format(type='torch')

    print(tokenized_dataset_train[6])


    # 对训练集和验证集应用过滤函数
    filtered_dataset_train = tokenized_dataset_train.filter(filter_short_samples)
    filtered_dataset_val = tokenized_dataset_val.filter(filter_short_samples)
    print(f"训练集过滤前样本数量: {len(tokenized_dataset_train)}")
    print(f"训练集过滤后样本数量: {len(filtered_dataset_train)}")
    print(f"验证集过滤前样本数量: {len(tokenized_dataset_val)}")
    print(f"验证集过滤后样本数量: {len(filtered_dataset_val)}")

    # 放入torch
    import torch
    filtered_dataset_train.set_format(type='torch')
    filtered_dataset_val.set_format(type='torch')
    print(filtered_dataset_train[0])
    print(filtered_dataset_val[0])


    return filtered_dataset_train, filtered_dataset_val, dataset["test"]

data_path = ["../SFT微调大模型/data_raw/train.jsonl","../SFT微调大模型/data_raw/valid.jsonl","../SFT微调大模型/data_raw/test.jsonl"]
tokenizer = AutoTokenizer.from_pretrained(model_path)
filtered_dataset_train, filtered_dataset_val, dataset_test = load_data(data_path=data_path,tokenizer=tokenizer)

from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
# 将 eos token 作为 pad token
tokenizer.pad_token = tokenizer.eos_token

# data_collator = DataCollatorWithPadding(tokenizer)
dataloader_params = {
    'batch_size': 16,
    'shuffle': True,
    'collate_fn': reward_collate_fn
}
train_dataloader = DataLoader(filtered_dataset_train, **dataloader_params)
val_dataloader = DataLoader(filtered_dataset_val, **dataloader_params)

batch = next(iter(train_dataloader))
print(batch.keys())

print(batch['input_ids'][1])
print(batch['attention_mask'][1])
print(batch['score'][1])
print(batch['score_index'][1])
print(tokenizer.decode(batch['input_ids'][1]))
print(batch['attention_mask'][1].nonzero()[-1])
